# Iceberg datalake migration from shared bucket to dedicated bucket


####  Run this cell to set up and start your interactive session.


In [21]:
%stop_session
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 300
%glue_version 5.0

# DEDICATED_BUCKET="jpmc-iceberg-whv2-038676220235-us-west-2"
%%configure
{
  "--datalake-formats": "iceberg",
  "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://jpmc-iceberg-whv2-038676220235-us-west-2/warehouse/ --conf spark.sql.catalog.glue_catalog.type=glue --conf spark.sql.defaultCatalog=glue_catalog"
}



Stopping session: 7ad88350-8b07-4301-b5d7-44e422528cb7
Stopped session.
Setting Glue version to: 5.0
Previous worker type: G.1X
Setting new worker type to: G.1X
Previous number of workers: 2
Setting new number of workers to: 2
Current idle_timeout is 300 minutes.
idle_timeout has been set to 300 minutes.
Setting Glue version to: 5.0
The following configurations have been updated: {'--datalake-formats': 'iceberg', '--conf': 'spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://jpmc-iceberg-whv2-038676220235-us-west-2/warehouse/ --conf spark.sql.catalog.glue_catalog.type=glue --conf spark.sql.defaultCatalog=glue_catalog'}


In [1]:


import sys
from pyspark.sql import SparkSession
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job


spark = SparkSession.builder.getOrCreate()

# Initialize Glue context
glueContext = GlueContext(spark.sparkContext)
job = Job(glueContext)

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 300
Session ID: 12266b10-1a7f-44a3-8b58-883f5cbe0fcb
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--datalake-formats iceberg
--conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://jpmc-iceberg-whv2-038676220235-us-west-2/warehouse/ --conf spark.sql.catalog.glue_catalog.type=glue --conf spark.sql.defaultCatalog=glue_catalog
Waiting for session 12266b10-1a7f-44a3-8b58-883f5cbe0fcb to get into ready status...
Session 12266b10-1a7f-44a3-8b58-883f5cbe0fcb has been created.



In [17]:
# Prelim: Find latest metadata file for each table
import boto3
import os
# replace with your table names
tables = ["main_rd_table", "main_sample_table"]  
metadata_files = {}
database = "jpmc_demo_warehouse"
db_s3_prefix = "warehouse"
DEDICATED_BUCKET="jpmc-iceberg-whv2-038676220235-us-west-2"

glue_client = boto3.client('glue', region_name='us-west-2')

for table_name in tables:
    response = glue_client.get_table(DatabaseName=database, Name=table_name)

    # For Iceberg tables, the metadata location is stored in table properties
    table = response['Table']

    # Check StorageDescriptor location
    # print(f"Table Location: {table['StorageDescriptor']['Location']}")

    # The metadata_location is in the table parameters
    parameters = table.get('Parameters', {})
    metadata_location = parameters.get('metadata_location', 'Not found')
    # print(f"Latest Metadata File: {metadata_location}")
    metadata_files[table_name] = os.path.basename(metadata_location)

for item in metadata_files:
    print(item, metadata_files[item])

main_rd_table 00000-e6c92ec9-46d3-4dcd-9886-fada923c465e.metadata.json
main_sample_table 00000-770409d0-314c-4db7-bdc5-be800507073a.metadata.json


In [18]:
# Step 1: Drop and re-create the Glue table pointing to the new location.
# For each table, register it from the new metadata location

for table_name in tables:
    metadata_location = f"s3://{DEDICATED_BUCKET}/{db_s3_prefix}/{table_name}/metadata/"

    # Find the latest metadata.json version    
    latest_metadata_file = metadata_files[table_name]


    print(f"DROPING TABLE  glue_catalog.{database}.{table_name}")
    spark.sql(f"DROP TABLE IF EXISTS glue_catalog.{database}.{table_name}")

    print(f"CALL to register_table => '{database}.{table_name}',")
    print(f"metadata_file => 's3://{DEDICATED_BUCKET}/{db_s3_prefix}/{table_name}/metadata/{latest_metadata_file}' )")
    
    spark.sql(f"""CALL glue_catalog.system.register_table(
            table => '{database}.{table_name}',
            metadata_file => 's3://{DEDICATED_BUCKET}/{db_s3_prefix}/{table_name}/metadata/{latest_metadata_file}')""")
    
    print(f"Registered: {database}.{table_name}")

DROPING TABLE  glue_catalog.jpmc_demo_warehouse.main_rd_table
DataFrame[]
CALL to egister_table => 'jpmc_demo_warehouse.main_rd_table',
metadata_file => 's3://jpmc-iceberg-whv2-038676220235-us-west-2/warehouse/main_rd_table/metadata/00000-e6c92ec9-46d3-4dcd-9886-fada923c465e.metadata.json' )
DataFrame[current_snapshot_id: bigint, total_records_count: bigint, total_data_files_count: bigint]
Registered: jpmc_demo_warehouse.main_rd_table
DROPING TABLE  glue_catalog.jpmc_demo_warehouse.main_sample_table
DataFrame[]
CALL to egister_table => 'jpmc_demo_warehouse.main_sample_table',
metadata_file => 's3://jpmc-iceberg-whv2-038676220235-us-west-2/warehouse/main_sample_table/metadata/00000-770409d0-314c-4db7-bdc5-be800507073a.metadata.json' )
DataFrame[current_snapshot_id: bigint, total_records_count: bigint, total_data_files_count: bigint]
Registered: jpmc_demo_warehouse.main_sample_table


In [19]:
# Step 2: Rewrite data files to update all internal path references.

for table_name in tables:
    spark.sql(f"""
        CALL glue_catalog.system.rewrite_data_files(
            table => '{database}.{table_name}'
        )
    """)
    print(f"Rewritten: {database}.{table_name}")

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int]
Rewritten: jpmc_demo_warehouse.main_rd_table
DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int]
Rewritten: jpmc_demo_warehouse.main_sample_table


In [ ]:
# Step 3: Clean up old metadata


for table_name in tables:
    # Expire old snapshots (keeps only the latest)
    spark.sql(f"""
        CALL glue_catalog.system.expire_snapshots(
            table => '{database}.{table_name}',
            retain_last => 1
        )
    """)

    # Remove orphan files from the old copy
    spark.sql(f"""
        CALL glue_catalog.system.remove_orphan_files(
            table => '{database}.{table_name}'
        )
    """)
    print(f"Cleaned up: {database}.{table_name}")

#### SQL Validation


In [ ]:
for table_name in tables:
    df = spark.sql(f"SELECT count(*) as cnt FROM glue_catalog.{database}.{table_name}")
    print(f"{table_name} => {df.collect()[0]['cnt']}")
